# Malaria Detection from Blood Smear Images
EfficientNetB0 transfer learning on the [Cell Images for Detecting Malaria](https://www.kaggle.com/datasets/iarunava/cell-images-for-detecting-malaria) dataset.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

The trained model is saved to your Google Drive. On the next run the notebook loads it and skips training.

In [ ]:
!pip install -q kagglehub

In [ ]:
import os, io, glob, json, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import kagglehub
from PIL import Image
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

SEED = 42
IMG_SIZE = 128
BATCH = 64

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 0. Connect Google Drive (model gets saved here)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/malaria_project"
os.makedirs(SAVE_DIR, exist_ok=True)

FINAL_MODEL = f"{SAVE_DIR}/malaria_model.keras"
HISTORY     = f"{SAVE_DIR}/history.json"
print("Saved model exists:", os.path.exists(FINAL_MODEL))

## 1. Download data

In [ ]:
DATA_DIR = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
print(DATA_DIR)

If this asks for Kaggle credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets (key icon on the left), or run `kagglehub.login()`.

In [ ]:
def collect(class_name):
    # dataset has a nested duplicate folder, dedupe by filename
    files = glob.glob(os.path.join(DATA_DIR, "**", class_name, "*.png"), recursive=True)
    return sorted({os.path.basename(f): f for f in files}.values())   # sorted -> same split every run

para = collect("Parasitized")
uninf = collect("Uninfected")
print(f"Parasitized: {len(para)} | Uninfected: {len(uninf)}")

paths = np.array(para + uninf)
labels = np.array([1] * len(para) + [0] * len(uninf))

In [ ]:
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    for row, (lst, name) in enumerate([(para, "Parasitized"), (uninf, "Uninfected")]):
        ax[row, i].imshow(plt.imread(random.choice(lst)))
        ax[row, i].set_title(name, fontsize=9)
        ax[row, i].axis("off")
plt.tight_layout(); plt.show()

## 2. Split and input pipeline

In [ ]:
x_train, x_tmp, y_train, y_tmp = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
x_val, x_test, y_val, y_test = train_test_split(
    x_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(len(x_train), len(x_val), len(x_test))

def load(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, tf.cast(label, tf.float32)   # keep 0-255, EfficientNet rescales inside

def make_ds(x, y, train=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if train:
        ds = ds.shuffle(len(x), seed=SEED)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(x_train, y_train, train=True)
val_ds   = make_ds(x_val, y_val)
test_ds  = make_ds(x_test, y_test)

## 3. Model

In [ ]:
# cells have no orientation so flips/rotations are safe; colour jitter covers stain differences
augment = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
], name="augment")

base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = models.Model(inputs, outputs)

metrics = ["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.Recall(name="recall")]

def cbs(name):
    return [
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
        callbacks.ModelCheckpoint(f"{SAVE_DIR}/{name}.keras", monitor="val_auc", mode="max", save_best_only=True),
    ]

## 4. Train (or load the saved model)
Skips training if `malaria_model.keras` already exists in Drive. To force retraining, delete that file from `MyDrive/malaria_project`.

Stage 1 trains only the new head, stage 2 fine-tunes the whole network at a low learning rate.

In [ ]:
if os.path.exists(FINAL_MODEL):
    print("Found saved model, loading it. Skipping training.")
    model = tf.keras.models.load_model(FINAL_MODEL, compile=False)
else:
    model.compile(optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=metrics)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=cbs("stage1"))

    base.trainable = True
    model.compile(optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=metrics)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cbs("malaria_best"))

    model.save(FINAL_MODEL)
    json.dump({
        "acc": h1.history["accuracy"] + h2.history["accuracy"],
        "val_acc": h1.history["val_accuracy"] + h2.history["val_accuracy"],
        "split": len(h1.history["accuracy"]),
    }, open(HISTORY, "w"))
    print("Model saved to", FINAL_MODEL)

## 5. Evaluate

In [ ]:
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["Uninfected", "Parasitized"], digits=4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_prob)
has_hist = os.path.exists(HISTORY)

fig, ax = plt.subplots(1, 3 if has_hist else 2, figsize=(16 if has_hist else 11, 4.5))

ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks([0, 1], ["Uninfected", "Parasitized"])
ax[0].set_yticks([0, 1], ["Uninfected", "Parasitized"])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual"); ax[0].set_title("Confusion matrix")
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha="center", va="center",
                   color="white" if cm[i, j] > cm.max() / 2 else "black")

ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], "--", c="gray")
ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].set_title("ROC curve")

if has_hist:
    h = json.load(open(HISTORY))
    ax[2].plot(h["acc"], label="train"); ax[2].plot(h["val_acc"], label="val")
    ax[2].axvline(h["split"] - 1, ls="--", c="gray")
    ax[2].set_title("Accuracy (dashed = fine-tuning starts)"); ax[2].legend()

plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/results.png", dpi=150); plt.show()

## 6. Sample test predictions

In [ ]:
idx = np.random.choice(len(x_test), 8, replace=False)
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for a, i in zip(ax.ravel(), idx):
    a.imshow(plt.imread(x_test[i]))
    pred = "Parasitized" if y_prob[i] >= 0.5 else "Uninfected"
    true = "Parasitized" if y_test[i] == 1 else "Uninfected"
    a.set_title(f"{pred} ({y_prob[i]:.2f})\ntrue: {true}", fontsize=9,
                color="green" if pred == true else "red")
    a.axis("off")
plt.tight_layout(); plt.show()

## 7. Test your own images
Run the cell, click **Choose files**, and select one or more cell images. Works best with single-cell crops like the ones in the dataset.

In [ ]:
from google.colab import files

def predict_pil(img):
    img = img.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype="float32")[None]
    return float(model.predict(arr, verbose=0)[0][0])

uploaded = files.upload()
n = len(uploaded)
fig, ax = plt.subplots(1, n, figsize=(3.5 * n, 4), squeeze=False)
for a, (name, data) in zip(ax[0], uploaded.items()):
    img = Image.open(io.BytesIO(data))
    p = predict_pil(img)
    infected = p >= 0.5
    a.imshow(img)
    a.set_title(f"{'Parasitized' if infected else 'Uninfected'}\n{(p if infected else 1 - p):.1%} confident",
                color="red" if infected else "green")
    a.axis("off")
plt.tight_layout(); plt.show()